# Assignment 1 - Implementation of combined CNNs
__Author__: Anna Söderbärg<br>
__Date__: 21 Sep 2023<br>

__Description__: This program uses two CNNs. The first one predicts domain, which is then implemented in the second CNN. The second CNN consists of a text and a domain branch and results in a binary classifier. The program predicts the following labels for test sequences and writes the result to a cvs file:<br>
1 - Human generated<br>
0 - Machine generated

In [ ]:
import pandas as pd
import json
from tensorflow import keras
from keras.models import Model
from keras.models import Sequential
from keras.preprocessing.sequence import pad_sequences
import numpy as np
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Input, Concatenate, Dense
from imblearn.over_sampling import SMOTE
import random
from sklearn.utils import shuffle

## Pathways for training and test data

In [ ]:
# Pathways of domain 1, domain 2 and test data files
pathway_domain1 = 'domain1_train.json'
pathway_domain2 = 'domain2_train.json'
pathway_test = 'test_set.json'

## Preprocessing training data

In [ ]:
# Read domain 1 and domain 2 data sets
df1 = pd.read_json(pathway_domain1, lines = True)
df2 = pd.read_json(pathway_domain2, lines = True)

### Formating and combining domain 1 and domain 2 data
Sample sequences are classified by domain according to:<br>
0 - computer generated domain 1<br>
1 - computer generated domain 2<br>
2 - human generated domain 1<br>
3 - human generated domain 2<br>

In [ ]:
# Adding a domain column to dataframes
df1['domain'] = 0 # Adding a domain colum to training set 1
df1.loc[df1['label'] == 1, 'domain'] = 2 # Change all human generated to class 2
df2['domain'] = 1 # Adding a domain column to training set 2
df2.loc[df2['label'] == 1, 'domain'] = 3 # Change all human generated to class 3

# Concatenate into a combined dataframe
df_train = pd.concat([df1, df2], ignore_index=True) # Combining set 1 and 2 to one set

#### Data characteristics

In [ ]:
print('Samples for each domain in combined df:')
print(df_train['domain'].value_counts())
print('Samples of each model in domain 2:')
print(df_train['model'].value_counts())

### Undersampling domain 2 models
As there is an overrepresenation of model 0, 1, 2, 3, and 6, samples of these models are removed. The number of samples removed is based on creating an equal amount of samples between class 0, 1, 2, and 3.

In [ ]:
# Undersample model 0, 1, 2, 3, 6
n_samples = 1636 # Maximum number of samples to keep for each model
undersampled_df = pd.DataFrame() # Dataframe to store undersampled data
for model in [0.0, 1.0, 2.0, 3.0, 6.0]:
    sampled_subset = df_train[df_train['model'] == model].sample(n=n_samples, random_state=42)
    undersampled_df = pd.concat([undersampled_df, sampled_subset], axis=0)

# Shuffle the undersampled data
undersampled_df = undersampled_df.sample(frac=1, random_state=42).reset_index(drop=True)

#### Combine undersampled data with domain 1 data and model 4, 5 and human from domain 2

In [ ]:
# Combine undersampled data with data not undersampled
df_resampled = pd.concat([df_train[df_train['model'] == 5.0],
                         df_train[df_train['model'] == 4.0],
                         df_train[df_train['domain'] == 2],
                         df_train[df_train['domain'] == 0],
                          df_train[df_train['domain'] == 3],
                         undersampled_df]
                        )

# Shuffle combined dataset
print('Samples of each class after undersampling:')
print(df_resampled['domain'].value_counts())

### Turning dataset into padded vectors
Each sample sequence is padded in order to create equal lenght vectors.

In [ ]:
# Checking lenght of the longest sequece
index = df_train['text'].apply(len).idxmax() # Index of longest sequence in training data
max_vector_size = len(df_train['text'][index]) # Nr of elements of longest sequence

# Padd sequence vectors into the lenght of the longest sequence
X_train = pad_sequences(df_resampled['text'], maxlen=max_vector_size)
y_domain = df_resampled['domain'] # Classes 0-3 of each sequence

### Oversampling human data samples
As the human generated data samples from domain 2 are underrepresented, oversampling is made using SMOTE

In [ ]:
# Resampling data using SMOTE
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(X_train, y_domain)
X_resampled, y_resampled = shuffle(X_resampled, y_resampled)

In [ ]:
df = pd.DataFrame(y_resampled)
print('Samples of each class after applying SMOTE:')
print(df['domain'].value_counts())

## CNN Model for classifying domain
Creating and training a model for classifying which class sequences belong to 0, 1, 2, or 3

In [ ]:
def create_domain_model(max_vector_size = 1075):
    """ Function creates and compiles a CNN model for classifying which model has generated a sequence
    Input: max sequence length
    Otput: created and compiled model """
    model = Sequential() 
    model.add(Embedding(input_dim=5000,
                               output_dim=180,
                               input_length=max_vector_size)) # Embedding layer
    model.add(Conv1D(180, 3, activation='relu')) # One dimensional convolusional layer
    model.add(GlobalMaxPooling1D()) # Global max pooling layer
    model.add(Dense(64, activation='relu')) # Fully connected layer
    model.add(Dense(5, activation='softmax')) # Output layer
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy']) # Compiling model
    return model

In [ ]:
def train_model(model, X, y):
    """ Trains a model based on training data with labels
    Input: Model, training data X, labels y
    Output: Trained model """
    model.fit(X, y, epochs=5, batch_size=32)
    return model

In [ ]:
# Creating CNN model for model prediction
model_domain = create_domain_model(max_vector_size)

# Training CNN model for model prediction
model_domain = train_model(model_domain, X_resampled, y_resampled)

## Binary classification model
Binary classes:

0 - machine generated

1 - human generated

### Creating binary label vector
Creating binary label vector based on resampled training data

In [ ]:
# Creating labels based on class 0-3 in resampled training data
y_label_resampled = [] # Storage of label values of sequences in training data
for i in y_resampled:
    if i == 2 or i == 3:
        y_label_resampled.append(1)
    else:
        y_label_resampled.append(0) 

# Reformating lists to numpy array to train binary classification model
y_resampled = np.array(y_resampled)
y_label_resampled = np.array(y_label_resampled)

### Binary CNN for classifying human/computer sequences
Model have one text branch and one model branch which handles classes 0-3 from domain 1 and 2

In [ ]:
embedding_dim = 10
num_models = df_train['domain'].nunique() # Number of classes 0-3

# Create a model
input_text = Input(shape=(max_vector_size,))
input_domain = Input(shape=(1,))

# Text branch
text_embedding = Embedding(input_dim=5000, output_dim=50, input_length=max_vector_size)(input_text)
text_conv = Conv1D(filters=128, kernel_size=3, activation='relu')(text_embedding)
text_pooled = GlobalMaxPooling1D()(text_conv)

# Domain branch
domain_embedding = Embedding(input_dim=num_models, output_dim=embedding_dim, input_length=1)(input_domain)
domain_flattened = GlobalMaxPooling1D()(domain_embedding)

# Concatenate
combined = Concatenate()([text_pooled, domain_flattened])

dense = Dense(10, activation='relu')(combined)
output = Dense(1, activation='sigmoid')(dense)

model = Model(inputs=[input_text, input_domain], outputs=output)

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Train the model
model.fit([X_resampled, y_resampled], y_label_resampled, epochs=10, batch_size=32)

## Prediction of test data

In [ ]:
# Read test data from json file
df_test = pd.read_json(pathway_test, lines = True)

# Padding test samples into equal length vectors
X_test = pad_sequences(df_test['text'], maxlen=max_vector_size)

### Classifying test data sequences in domains

In [ ]:
# Predicting which class sequences come from 0-3
pred_prob = model_domain.predict(X_test)

# Storing predicted class labels
class_labels = []
for i in pred_prob:
    label = np.argmax(i)
    class_labels.append(label)

### Performing binary classification

In [ ]:
df_class_labels = pd.DataFrame(class_labels)
print('predicted classes for test data:')
print(df_class_labels.value_counts())

### Classifying if test data sequence is computer generated

In [ ]:
# Adding the predicted generated model to padded sequence vector
X_test = pad_sequences(df_test['text'], maxlen=max_vector_size)
class_labels = np.array(class_labels)

# Using binary model to predict label 0 or 1
pred_prob = model.predict([X_test, class_labels])
class_labels_binary = (pred_prob >= 0.5).astype(int).flatten().tolist()

In [ ]:
df_test = pd.DataFrame(class_labels_binary)
print('Predicted labels for test data:')
print(df_test.value_counts())

## Write result to file

In [ ]:
file = open("output.csv", "w")
id = 0
file.write('id,class\n')
for point in class_labels_binary:
    file.write(str(id) +','+ str(point)+'\n')
    id += 1
file.close()